# PharmaKinetics: Обучение GIN-моделей на A100

Полный пайплайн обучения и сравнения GIN-моделей для предсказания токсичности.

**Модели:**
1. GIN scratch (baseline)
2. GIN + Virtual Node
3. GIN + OGB features + Virtual Node
4. GIN + VN + Attention pooling
5. GIN + VN + Focal loss
6. GIN pretrained + VN + XAI loss

**Оценка:** 3 seeds × scaffold split → mean ± std ROC-AUC

**Runtime:** ~15-20 мин на A100

## 1. Установка зависимостей и клонирование репозитория

In [ ]:
import os

REPO_DIR = "/content/admet"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Isk2x/admet.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Рабочая директория: {os.getcwd()}")
!ls src/

In [ ]:
import torch

# Определяем версию PyTorch и CUDA для правильных wheel
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

WHEEL_URL = f"https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html"
print(f"PyG wheels: {WHEEL_URL}")

!pip install -q torch-geometric rdkit pandas numpy scikit-learn xgboost matplotlib tqdm
!pip install -q torch_scatter torch_sparse torch_cluster -f {WHEEL_URL}

print(f"\nПроверка:")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

import torch_geometric
print(f"  PyG: {torch_geometric.__version__}")

## 2. Загрузка данных

In [ ]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.data.loader_multitask import load_tox21_multitask, TOX21_TASKS
from src.data.splitter import scaffold_split
from src.data.featurizer_gin import build_gin_dataset

df = load_tox21_multitask()
print(f"\nМолекул: {len(df)}, Задач: {len(TOX21_TASKS)}")

# Проверка scaffold split (balanced allocation)
train_df, val_df, test_df = scaffold_split(df, seed=42)
print(f"Scaffold split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

## 3. Обучение всех моделей (3 seeds)

In [ ]:
import time
import json
import numpy as np
from pathlib import Path
from functools import partial

from src.utils.seed import set_seed
from src.models.gin_pretrained import train_gin, download_pretrained
from src.models.xai_loss import combined_xai_loss, compute_toxicophore_loss

SEEDS = [42, 0, 1]

MODEL_CONFIGS = [
    {
        "name": "gin_scratch",
        "backbone_type": "standard",
        "pool_type": "mean",
        "pretrained": None,
        "use_focal_loss": False,
        "xai_lambda": 0.0,
        "ogb_features": False,
    },
    {
        "name": "gin_vn",
        "backbone_type": "vn",
        "pool_type": "mean",
        "pretrained": None,
        "use_focal_loss": False,
        "xai_lambda": 0.0,
        "ogb_features": False,
    },
    {
        "name": "gin_ogb_vn",
        "backbone_type": "ogb_vn",
        "pool_type": "mean",
        "pretrained": None,
        "use_focal_loss": False,
        "xai_lambda": 0.0,
        "ogb_features": True,
    },
    {
        "name": "gin_vn_attn",
        "backbone_type": "vn",
        "pool_type": "attention",
        "pretrained": None,
        "use_focal_loss": False,
        "xai_lambda": 0.0,
        "ogb_features": False,
    },
    {
        "name": "gin_vn_focal",
        "backbone_type": "vn",
        "pool_type": "mean",
        "pretrained": None,
        "use_focal_loss": True,
        "xai_lambda": 0.0,
        "ogb_features": False,
    },
    {
        "name": "gin_pretrained_vn_xai",
        "backbone_type": "vn",
        "pool_type": "mean",
        "pretrained": "supervised_contextpred",
        "use_focal_loss": False,
        "xai_lambda": 0.1,
        "ogb_features": False,
    },
    # === НОВЫЕ ИННОВАЦИИ ===
    {
        "name": "gin_vn_uncmtl",
        "backbone_type": "vn",
        "pool_type": "mean",
        "pretrained": None,
        "use_focal_loss": False,
        "use_uncertainty_loss": True,
        "xai_lambda": 0.0,
        "toxicophore_lambda": 0.0,
        "ogb_features": False,
    },
    {
        "name": "gin_vn_toxguided",
        "backbone_type": "vn",
        "pool_type": "mean",
        "pretrained": None,
        "use_focal_loss": False,
        "use_uncertainty_loss": False,
        "xai_lambda": 0.1,
        "toxicophore_lambda": 0.05,
        "ogb_features": False,
    },
    {
        "name": "gin_pretrained_vn_full",
        "backbone_type": "vn",
        "pool_type": "mean",
        "pretrained": "supervised_contextpred",
        "use_focal_loss": False,
        "use_uncertainty_loss": True,
        "xai_lambda": 0.1,
        "toxicophore_lambda": 0.05,
        "ogb_features": False,
    },
]

# Скачиваем pretrained веса
download_pretrained("supervised_contextpred")

all_seed_results = {}
start_total = time.time()

for seed in SEEDS:
    print(f"\n{'#'*70}")
    print(f"# SEED = {seed}")
    print(f"{'#'*70}")
    
    set_seed(seed)
    train_df, val_df, test_df = scaffold_split(df, seed=seed)
    
    train_std = build_gin_dataset(train_df, TOX21_TASKS, ogb_features=False)
    val_std = build_gin_dataset(val_df, TOX21_TASKS, ogb_features=False)
    test_std = build_gin_dataset(test_df, TOX21_TASKS, ogb_features=False)
    
    train_ogb = build_gin_dataset(train_df, TOX21_TASKS, ogb_features=True)
    val_ogb = build_gin_dataset(val_df, TOX21_TASKS, ogb_features=True)
    test_ogb = build_gin_dataset(test_df, TOX21_TASKS, ogb_features=True)
    
    seed_results = {}
    
    for cfg in MODEL_CONFIGS:
        set_seed(seed)
        start_model = time.time()
        
        if cfg["ogb_features"]:
            td, vd, ted = train_ogb, val_ogb, test_ogb
        else:
            td, vd, ted = train_std, val_std, test_std
        
        xai_fn = None
        if cfg["xai_lambda"] > 0:
            xai_fn = partial(combined_xai_loss, faith_weight=0.5, stab_weight=0.5)
        
        tox_fn = None
        tox_lambda = cfg.get("toxicophore_lambda", 0.0)
        if tox_lambda > 0:
            tox_fn = compute_toxicophore_loss
        
        results, _ = train_gin(
            td, vd, ted,
            num_tasks=len(TOX21_TASKS),
            task_names=TOX21_TASKS,
            pretrained=cfg["pretrained"],
            xai_loss_fn=xai_fn,
            xai_lambda=cfg["xai_lambda"],
            lr=1e-3,
            epochs=100,
            patience=15,
            batch_size=64,
            model_name=f"{cfg['name']}_s{seed}",
            backbone_type=cfg["backbone_type"],
            pool_type=cfg["pool_type"],
            use_focal_loss=cfg["use_focal_loss"],
            use_uncertainty_loss=cfg.get("use_uncertainty_loss", False),
            toxicophore_loss_fn=tox_fn,
            toxicophore_lambda=tox_lambda,
        )
        
        elapsed = time.time() - start_model
        print(f"  ⏱ {cfg['name']}: {elapsed:.0f} сек")
        seed_results[cfg["name"]] = results
    
    all_seed_results[seed] = seed_results

total_time = time.time() - start_total
print(f"\nОбщее время: {total_time/60:.1f} мин")

## 4. Агрегация результатов

In [ ]:
model_names = [c["name"] for c in MODEL_CONFIGS]
aggregated = {}

for mname in model_names:
    test_aucs = []
    val_aucs = []
    test_praucs = []
    for seed in SEEDS:
        if mname in all_seed_results[seed]:
            res = all_seed_results[seed][mname]
            test_aucs.append(res["test"]["mean_roc_auc"])
            val_aucs.append(res["val"]["mean_roc_auc"])
            test_praucs.append(res["test"]["mean_pr_auc"])
    
    aggregated[mname] = {
        "test_roc_auc": f"{np.mean(test_aucs):.4f} ± {np.std(test_aucs):.4f}",
        "test_pr_auc": f"{np.mean(test_praucs):.4f} ± {np.std(test_praucs):.4f}",
        "val_roc_auc": f"{np.mean(val_aucs):.4f} ± {np.std(val_aucs):.4f}",
        "_test_roc_auc_mean": float(np.mean(test_aucs)),
    }

print(f"{'='*86}")
print("СВОДКА РЕЗУЛЬТАТОВ (Tox21, scaffold split, 3 seeds)")
print(f"{'='*86}")
print(f"{'Модель':<30} {'Test ROC-AUC':>18} {'Test PR-AUC':>18} {'Val ROC-AUC':>18}")
print("-" * 86)
for mname in model_names:
    a = aggregated[mname]
    print(f"{mname:<30} {a['test_roc_auc']:>18} {a['test_pr_auc']:>18} {a['val_roc_auc']:>18}")
print("-" * 86)
print(f"{'Hu et al. 2020 SOTA':<30} {'0.7512 ± 0.0079':>18}")

## 5. XAI сравнение (AOPC / IoU)

In [ ]:
from src.explain.atom_importance_gin import compute_atom_importance_gin, get_top_k_atoms_gin
from src.models.gin_pretrained import GINMultiTask
from src.data.featurizer_gin import smiles_to_graph_gin
from torch_geometric.nn import global_mean_pool as _gmp

set_seed(42)
train_df, val_df, test_df = scaffold_split(df, seed=42)
test_data_std = build_gin_dataset(test_df, TOX21_TASKS, ogb_features=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
fraction = 0.2
max_molecules = 200

xai_models_to_compare = ["gin_scratch", "gin_vn", "gin_pretrained_vn_xai"]
xai_backbone_types = {"gin_scratch": "standard", "gin_vn": "vn", "gin_pretrained_vn_xai": "vn"}

xai_results = {}

for model_name in xai_models_to_compare:
    bt = xai_backbone_types[model_name]
    
    # Берём модель из seed=42
    if model_name not in all_seed_results.get(42, {}):
        print(f"  [{model_name}] Нет результатов для seed 42")
        continue
    
    model = GINMultiTask(num_tasks=len(TOX21_TASKS), backbone_type=bt)
    # Переобучаем для XAI (используем тот же seed=42)
    set_seed(42)
    xai_fn = None
    pretrained = None
    for cfg in MODEL_CONFIGS:
        if cfg["name"] == model_name:
            pretrained = cfg["pretrained"]
            if cfg["xai_lambda"] > 0:
                xai_fn = partial(combined_xai_loss, faith_weight=0.5, stab_weight=0.5)
            break
    
    td = build_gin_dataset(train_df, TOX21_TASKS, ogb_features=False)
    vd = build_gin_dataset(val_df, TOX21_TASKS, ogb_features=False)
    ted = build_gin_dataset(test_df, TOX21_TASKS, ogb_features=False)
    
    _, model = train_gin(
        td, vd, ted,
        num_tasks=len(TOX21_TASKS),
        task_names=TOX21_TASKS,
        pretrained=pretrained,
        xai_loss_fn=xai_fn,
        xai_lambda=0.1 if xai_fn else 0.0,
        lr=1e-3,
        epochs=100,
        patience=15,
        batch_size=64,
        model_name=f"{model_name}_xai_eval",
        backbone_type=bt,
    )
    model = model.to(device)
    model.eval()
    
    n = min(len(test_data_std), max_molecules)
    aopc_results = []
    
    for i in range(n):
        data = test_data_std[i]
        if data.x.size(0) < 3:
            continue
        
        imp = compute_atom_importance_gin(model, data, device)
        k = max(1, int(len(imp) * fraction))
        top_idx = np.argsort(imp)[-k:]
        
        data_dev = data.clone().to(device)
        data_dev.batch = torch.zeros(data_dev.x.size(0), dtype=torch.long, device=device)
        
        with torch.no_grad():
            original_pred = torch.sigmoid(model(data_dev)).mean().item()
            h = model.backbone(data_dev.x, data_dev.edge_index, data_dev.edge_attr) if bt == 'standard' else model.backbone(data_dev.x, data_dev.edge_index, data_dev.edge_attr, data_dev.batch)
            h_masked = h.clone()
            for idx in top_idx:
                h_masked[idx] = 0.0
            graph_repr = _gmp(h_masked, data_dev.batch)
            masked_pred = torch.sigmoid(model.classifier(graph_repr)).mean().item()
        
        aopc_results.append(original_pred - masked_pred)
    
    mean_aopc = float(np.mean(aopc_results)) if aopc_results else 0.0
    xai_results[model_name] = {"mean_aopc": mean_aopc, "n_molecules": len(aopc_results)}
    print(f"  {model_name}: AOPC = {mean_aopc:.4f} ({len(aopc_results)} молекул)")

print(f"\nСравнение AOPC:")
for name, r in xai_results.items():
    print(f"  {name}: {r['mean_aopc']:.4f}")

## 6. Дополнительные датасеты (ClinTox, BBBP)

In [ ]:
from src.experiments.run_extra_datasets import run_experiment as run_extra

extra_results = run_extra()

## 7. Сохранение артефактов

In [ ]:
import json
from pathlib import Path

ARTIFACTS_DIR = Path(REPO_DIR) / "artifacts" / "metrics"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Сохраняем основные результаты
full_results = {
    "tox21_aggregated": {k: {kk: vv for kk, vv in v.items() if not kk.startswith('_')} for k, v in aggregated.items()},
    "xai_comparison": xai_results,
    "training_time_minutes": round(total_time / 60, 1),
}

with open(ARTIFACTS_DIR / "e4_colab_results.json", "w") as f:
    json.dump(full_results, f, indent=2, default=float)

print(f"Результаты сохранены в {ARTIFACTS_DIR / 'e4_colab_results.json'}")

# Копирование в Google Drive (опционально)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    drive_dir = Path('/content/drive/MyDrive/PharmaKinetics')
    drive_dir.mkdir(parents=True, exist_ok=True)
    
    import shutil
    shutil.copy(ARTIFACTS_DIR / 'e4_colab_results.json', drive_dir / 'e4_colab_results.json')
    print(f"Скопировано в Google Drive: {drive_dir}")
except Exception as e:
    print(f"Google Drive недоступен: {e}")
    print("Результаты сохранены локально.")

## 8. Fine-tuning ChemBERTa-2 (Transformer, ~80M params)

Более мощная модель на основе RoBERTa, обученная на 77M молекул из PubChem.
Работает напрямую со SMILES (без графовой конвертации).

**Ожидаемое время:** ~10-15 мин на A100

In [ ]:
!pip install -q transformers

from transformers import AutoTokenizer
print("transformers установлен")

from src.models.chemberta import CHEMBERTA_MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(CHEMBERTA_MODEL_NAME)
print(f"Токенизатор загружен: {CHEMBERTA_MODEL_NAME}")

In [ ]:
import time
from src.models.chemberta import train_chemberta

CHEMBERTA_SEEDS = [42, 0, 1]
MODELS_DIR_CB = Path(REPO_DIR) / "artifacts" / "models"

chemberta_all_results = []
start_cb = time.time()

for seed in CHEMBERTA_SEEDS:
    print(f"\n{'#'*60}")
    print(f"# ChemBERTa Seed = {seed}")
    print(f"{'#'*60}")

    set_seed(seed)
    train_df_cb, val_df_cb, test_df_cb = scaffold_split(df, seed=seed)

    train_smiles = train_df_cb["smiles"].tolist()
    val_smiles = val_df_cb["smiles"].tolist()
    test_smiles = test_df_cb["smiles"].tolist()

    train_labels = train_df_cb[TOX21_TASKS].values.astype(float)
    val_labels = val_df_cb[TOX21_TASKS].values.astype(float)
    test_labels = test_df_cb[TOX21_TASKS].values.astype(float)

    results_cb, model_cb = train_chemberta(
        train_smiles, train_labels,
        val_smiles, val_labels,
        test_smiles, test_labels,
        num_tasks=12,
        task_names=TOX21_TASKS,
        lr=2e-5,
        epochs=20,
        patience=5,
        batch_size=32,
        device=device,
        save_dir=str(MODELS_DIR_CB) if seed == 42 else None,
        model_name=f"chemberta_tox21_s{seed}",
    )
    chemberta_all_results.append(results_cb)

cb_time = time.time() - start_cb
print(f"\nChemBERTa обучение завершено: {cb_time/60:.1f} мин")

In [ ]:
cb_test_aucs = [r["test"]["mean_roc_auc"] for r in chemberta_all_results]
cb_val_aucs = [r["val"]["mean_roc_auc"] for r in chemberta_all_results]

print(f"{'='*60}")
print("ChemBERTa-2 vs GIN — ИТОГОВОЕ СРАВНЕНИЕ")
print(f"{'='*60}")
print(f"{'Модель':<30} {'Test ROC-AUC':>20}")
print("-" * 52)

for mname in [c["name"] for c in MODEL_CONFIGS]:
    a = aggregated[mname]
    print(f"{mname:<30} {a['test_roc_auc']:>20}")

cb_line = f"{np.mean(cb_test_aucs):.4f} ± {np.std(cb_test_aucs):.4f}"
print(f"{'chemberta_77m':<30} {cb_line:>20}")
print("-" * 52)
print(f"{'Hu et al. 2020 SOTA':<30} {'0.7512 ± 0.0079':>20}")
print(f"{'='*60}")

# Сохраняем результаты ChemBERTa
cb_summary = {
    "model": "ChemBERTa-77M-MLM",
    "dataset": "Tox21",
    "split": "scaffold",
    "seeds": CHEMBERTA_SEEDS,
    "test_roc_auc_mean": float(np.mean(cb_test_aucs)),
    "test_roc_auc_std": float(np.std(cb_test_aucs)),
    "val_roc_auc_mean": float(np.mean(cb_val_aucs)),
    "val_roc_auc_std": float(np.std(cb_val_aucs)),
    "training_time_minutes": round(cb_time / 60, 1),
    "per_seed_results": chemberta_all_results,
}

with open(ARTIFACTS_DIR / "e6_chemberta_results.json", "w") as f:
    json.dump(cb_summary, f, indent=2, default=float)

print(f"\nРезультаты: {ARTIFACTS_DIR / 'e6_chemberta_results.json'}")

# Копируем веса для MVP
try:
    from google.colab import files
    weights_path = MODELS_DIR_CB / "chemberta_tox21_s42_weights.pt"
    if weights_path.exists():
        files.download(str(weights_path))
        print("Веса ChemBERTa скачаны для MVP")
except Exception:
    print("Веса сохранены локально:", MODELS_DIR_CB)

## 9. 3D-модели: SchNet + Hybrid GIN-SchNet Fusion

Настоящие 3D GNN — SchNet работает с атомными координатами (ETKDG conformers),
строит radius graph (cutoff=10Å) и использует continuous-filter convolutions.

**Модели:**
1. SchNet standalone (single conformer)
2. MultiConf-SchNet (5 конформеров, attention aggregation)
3. Hybrid GIN(2D) + SchNet(3D) — attention gate fusion
4. Hybrid + Uncertainty-Weighted MTL

**Ожидаемое время:** ~30-40 мин на A100

In [ ]:
import time
import numpy as np
from src.data.featurizer_3d import build_3d_dataset, build_3d_multi_dataset, smiles_to_3d
from src.data.featurizer_gin import smiles_to_graph_gin
from src.models.schnet_model import train_schnet, train_multiconf_schnet
from src.models.hybrid_fusion import build_paired_dataset, train_hybrid
from src.models.gin_pretrained import download_pretrained

download_pretrained("supervised_contextpred")

SEEDS_3D = [42, 0, 1]
all_3d_results = {}
start_3d = time.time()

for seed in SEEDS_3D:
    print(f"\n{'#'*70}")
    print(f"# 3D SEED = {seed}")
    print(f"{'#'*70}")

    set_seed(seed)
    train_df_3d, val_df_3d, test_df_3d = scaffold_split(df, seed=seed)

    seed_res = {}

    # --- SchNet standalone ---
    print("\n--- SchNet standalone ---")
    set_seed(seed)
    train_3d = build_3d_dataset(train_df_3d, TOX21_TASKS, seed=seed)
    val_3d = build_3d_dataset(val_df_3d, TOX21_TASKS, seed=seed)
    test_3d = build_3d_dataset(test_df_3d, TOX21_TASKS, seed=seed)
    print(f"  3D data: train={len(train_3d)}, val={len(val_3d)}, test={len(test_3d)}")

    schnet_res, _ = train_schnet(
        train_3d, val_3d, test_3d,
        num_tasks=len(TOX21_TASKS), task_names=TOX21_TASKS,
        hidden_channels=128, num_interactions=6, cutoff=10.0,
        lr=1e-3, epochs=100, patience=15, batch_size=64,
        model_name=f"schnet_s{seed}",
    )
    seed_res["schnet"] = schnet_res

    # --- MultiConf-SchNet ---
    print("\n--- MultiConf-SchNet (5 conformers) ---")
    set_seed(seed)
    train_mc = build_3d_multi_dataset(train_df_3d, TOX21_TASKS, num_confs=5, seed=seed)
    val_mc = build_3d_multi_dataset(val_df_3d, TOX21_TASKS, num_confs=5, seed=seed)
    test_mc = build_3d_multi_dataset(test_df_3d, TOX21_TASKS, num_confs=5, seed=seed)
    print(f"  MultiConf: train={len(train_mc)}, val={len(val_mc)}, test={len(test_mc)}")

    mc_res, _ = train_multiconf_schnet(
        train_mc, val_mc, test_mc,
        num_tasks=len(TOX21_TASKS), task_names=TOX21_TASKS,
        hidden_channels=128, num_interactions=6, cutoff=10.0,
        lr=1e-3, epochs=60, patience=10,
        model_name=f"schnet_multiconf_s{seed}",
    )
    seed_res["schnet_multiconf"] = mc_res

    # --- Hybrid GIN + SchNet ---
    print("\n--- Hybrid GIN(2D) + SchNet(3D) ---")
    set_seed(seed)
    train_2d_h, train_3d_h = build_paired_dataset(
        train_df_3d, TOX21_TASKS, smiles_to_graph_gin,
        lambda s, y: smiles_to_3d(s, y, seed=seed),
    )
    val_2d_h, val_3d_h = build_paired_dataset(
        val_df_3d, TOX21_TASKS, smiles_to_graph_gin,
        lambda s, y: smiles_to_3d(s, y, seed=seed),
    )
    test_2d_h, test_3d_h = build_paired_dataset(
        test_df_3d, TOX21_TASKS, smiles_to_graph_gin,
        lambda s, y: smiles_to_3d(s, y, seed=seed),
    )
    print(f"  Paired: train={len(train_2d_h)}, val={len(val_2d_h)}, test={len(test_2d_h)}")

    hybrid_res, _ = train_hybrid(
        train_2d_h, train_3d_h, val_2d_h, val_3d_h, test_2d_h, test_3d_h,
        num_tasks=len(TOX21_TASKS), task_names=TOX21_TASKS,
        pretrained="supervised_contextpred",
        lr=1e-3, backbone_lr=1e-4,
        epochs=100, patience=15, batch_size=64,
        model_name=f"hybrid_gin_schnet_s{seed}",
    )
    seed_res["hybrid_gin_schnet"] = hybrid_res

    # --- Hybrid + UncMTL ---
    print("\n--- Hybrid + Uncertainty-Weighted MTL ---")
    set_seed(seed)
    hybrid_unc_res, _ = train_hybrid(
        train_2d_h, train_3d_h, val_2d_h, val_3d_h, test_2d_h, test_3d_h,
        num_tasks=len(TOX21_TASKS), task_names=TOX21_TASKS,
        pretrained="supervised_contextpred",
        lr=1e-3, backbone_lr=1e-4,
        epochs=100, patience=15, batch_size=64,
        model_name=f"hybrid_gin_schnet_unc_s{seed}",
        use_uncertainty_loss=True,
    )
    seed_res["hybrid_gin_schnet_unc"] = hybrid_unc_res

    all_3d_results[seed] = seed_res

total_3d_time = time.time() - start_3d
print(f"\n3D эксперименты завершены: {total_3d_time/60:.1f} мин")

## 10. Итоговое сравнение: 2D vs 3D vs Hybrid

In [ ]:
model_names_3d = ["schnet", "schnet_multiconf", "hybrid_gin_schnet", "hybrid_gin_schnet_unc"]
agg_3d = {}

for mname in model_names_3d:
    test_aucs = []
    val_aucs = []
    test_praucs = []
    for seed in SEEDS_3D:
        if mname in all_3d_results.get(seed, {}):
            res = all_3d_results[seed][mname]
            test_aucs.append(res["test"]["mean_roc_auc"])
            val_aucs.append(res["val"]["mean_roc_auc"])
            test_praucs.append(res["test"]["mean_pr_auc"])

    agg_3d[mname] = {
        "test_roc_auc": f"{np.mean(test_aucs):.4f} ± {np.std(test_aucs):.4f}",
        "test_pr_auc": f"{np.mean(test_praucs):.4f} ± {np.std(test_praucs):.4f}",
        "val_roc_auc": f"{np.mean(val_aucs):.4f} ± {np.std(val_aucs):.4f}",
        "_test_roc_auc_mean": float(np.mean(test_aucs)),
    }

print(f"{'='*90}")
print("ИТОГОВОЕ СРАВНЕНИЕ: 2D GIN vs 3D SchNet vs Hybrid (Tox21, scaffold, 3 seeds)")
print(f"{'='*90}")
print(f"{'Модель':<30} {'Test ROC-AUC':>18} {'Test PR-AUC':>18} {'Val ROC-AUC':>18}")
print("-" * 86)

# GIN модели из E4
for mname in [c["name"] for c in MODEL_CONFIGS]:
    a = aggregated[mname]
    print(f"{mname:<30} {a['test_roc_auc']:>18} {a['test_pr_auc']:>18} {a['val_roc_auc']:>18}")

print("-" * 86)

# 3D модели
for mname in model_names_3d:
    a = agg_3d[mname]
    print(f"{mname:<30} {a['test_roc_auc']:>18} {a['test_pr_auc']:>18} {a['val_roc_auc']:>18}")

# ChemBERTa
cb_line = f"{np.mean(cb_test_aucs):.4f} ± {np.std(cb_test_aucs):.4f}"
cb_pr_line = "N/A"
cb_val_line = f"{np.mean(cb_val_aucs):.4f} ± {np.std(cb_val_aucs):.4f}"
print("-" * 86)
print(f"{'chemberta_77m':<30} {cb_line:>18} {cb_pr_line:>18} {cb_val_line:>18}")
print("-" * 86)
print(f"{'Hu et al. 2020 SOTA':<30} {'0.7512 ± 0.0079':>18}")
print(f"{'='*90}")

# Сохраняем все результаты
full_3d_results = {
    "3d_models": {k: {kk: vv for kk, vv in v.items() if not kk.startswith('_')}
                  for k, v in agg_3d.items()},
    "training_time_minutes": round(total_3d_time / 60, 1),
    "per_seed": {str(s): all_3d_results[s] for s in SEEDS_3D},
}

with open(ARTIFACTS_DIR / "e7_3d_comparison.json", "w") as f:
    json.dump(full_3d_results, f, indent=2, default=float)

print(f"\n3D результаты: {ARTIFACTS_DIR / 'e7_3d_comparison.json'}")

# Скачиваем артефакты
try:
    from google.colab import files
    files.download(str(ARTIFACTS_DIR / "e7_3d_comparison.json"))
    files.download(str(ARTIFACTS_DIR / "e4_colab_results.json"))
except Exception:
    print("Артефакты сохранены локально")